# CU-BEMS data preparation: clean, add weather and holidays

IS4001 Smart Building Analytics project.

This script turns the 14 raw CU-BEMS CSVs (one per floor per year, 1-minute data) into
clean, analysis-ready tables, with Bangkok weather and Thai public holidays attached.

**What it does, in order**

1. Loads each raw file, fixes timestamps (bad, duplicate and missing minutes) and renames columns.
2. Cleans at the raw 1-minute level, before any averaging:
   - power: removes negative values and physically impossible spikes;
   - temperature / humidity / lux: removes out-of-range readings and "stuck" sensors (flatlines);
   - fills short gaps (30 minutes or less) by interpolation and leaves longer gaps empty.
3. Aggregates to 15-minute and hourly tables. A window is only kept if enough real readings exist.
4. Adds derived columns: total AC power per zone, floor totals, indoor humidity ratio.
5. Downloads hourly Bangkok weather from the Open-Meteo historical API (cached after the first run).
6. Adds a calendar: Thai public holidays, weekday/weekend/holiday, occupied hours.
7. Joins 2018 and 2019 into one continuous table per floor and writes CSV files,
   plus a cleaning log and a data-quality report.

**Outputs** (in `OUT_DIR`)

| File | Contents |
|---|---|
| `clean_15min/floor{n}_15min.csv` | Clean 15-minute data per floor, with weather and calendar |
| `clean_hourly/floor{n}_hourly.csv` | Clean hourly data per floor, with weather and calendar (use for the baseline model) |
| `all_floors_hourly.csv` | All floors side by side (columns prefixed `f1_`, `f2_`...), hourly |
| `weather/openmeteo_bangkok_hourly.csv` | Raw weather download (cache) |
| `weather/daily_weather_cdd.csv` | Daily mean temperature and cooling degree days |
| `calendar/thai_holidays.csv` | Holiday list used, to cross-check against the Bank of Thailand list |
| `reports/cleaning_log.csv` | Every cleaning rule applied and how many readings it changed |
| `reports/data_quality.csv` | Missing data per file and column, before and after cleaning |
| `reports/idle_units.csv` | AC units that are effectively never on |

**Requirements:** `pip install pandas numpy requests holidays` (optional: `pyarrow` for Parquet copies).

**Data credits:** weather data by Open-Meteo.com (CC BY 4.0), based on ECMWF ERA5 reanalysis.
Cite it in your slides.

## 1. Settings
Change `RAW_DIR` to the folder holding the 14 original CSVs. Everything else can stay as is;
each threshold is a documented assumption you can quote in the methods slide.

In [4]:
!pip install holidays
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
import requests
import holidays

# --- Folders ---------------------------------------------------------------
RAW_DIR = Path("../../data/qns2")          # folder with 2018Floor1.csv ... 2019Floor7.csv
OUT_DIR = Path("../../data/qns2/processed")    # created if missing

# --- Location and period ------------------------------------------------------
# CU-BEMS building: Chulalongkorn University campus, central Bangkok (approximate).
LATITUDE, LONGITUDE = 13.7367, 100.5323
TIMEZONE = "Asia/Bangkok"           # dataset timestamps are local time (UTC+7, no daylight saving)
START_DATE, END_DATE = "2018-07-01", "2019-12-31"

# --- Cleaning rules (all applied at the raw 1-minute level) -------------------
POWER_CAP_QUANTILE = 0.999          # a power reading is a spike if it exceeds
POWER_CAP_MULTIPLIER = 3.0          #   max(3 x the column's 99.9th percentile, 10 kW)
POWER_CAP_FLOOR_KW = 10.0           # the 10 kW floor protects rarely used units from being cut
TEMP_RANGE_C = (10.0, 40.0)         # plausible indoor air temperature
RH_RANGE_PCT = (0.0, 100.0)         # relative humidity must be 0-100 %
LUX_RANGE = (0.0, 100_000.0)        # illuminance cannot be negative
FLATLINE_HOURS = 6                  # a T/RH/lux sensor repeating the exact same value for
                                    #   6 h or more is treated as stuck
SHORT_GAP_MINUTES = 30              # gaps up to 30 min are interpolated; longer gaps stay empty
MIN_COVERAGE = 0.67                 # a 15-min / hourly value needs >= 67 % of its readings present

# --- Analysis definitions -----------------------------------------------------
IDLE_AC_MEAN_KW = 0.1               # AC unit averaging below this is flagged as idle
OCC_START_HOUR, OCC_END_HOUR = 8, 18   # occupied = workday 08:00-17:59
CDD_BASE_C = 24.0                   # base temperature for cooling degree days / hours

# Add any special government days off that the `holidays` package misses, after checking
# them against the Bank of Thailand holiday list, e.g. {"2019-07-29": "Special holiday"}.
EXTRA_HOLIDAYS: dict[str, str] = {}

# --- Weather request ----------------------------------------------------------
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
# Open-Meteo variable name -> our column name
WEATHER_VARS = {
    "temperature_2m": "out_temp_c",
    "relative_humidity_2m": "out_rh_pct",
    "dew_point_2m": "out_dewpoint_c",
    "apparent_temperature": "out_apparent_temp_c",
    "precipitation": "out_precip_mmh",
    "cloud_cover": "out_cloud_pct",
    "shortwave_radiation": "out_solar_wm2",
    "wind_speed_10m": "out_wind_kmh",
}
# These two are totals/averages over the PRECEDING hour in Open-Meteo, so they are
# re-labelled to the start of that hour to line up with our hour-start building data.
PRECEDING_HOUR_VARS = ["out_precip_mmh", "out_solar_wm2"]

## 2. Helper functions: loading and renaming

In [5]:
FILE_PATTERN = re.compile(r"(\d{4})\s*Floor\s*(\d+)", re.IGNORECASE)
COLUMN_PATTERN = re.compile(r"^(z\d+)_([A-Za-z0-9]+)\((kW|degC|RH%|lux)\)$")


def find_raw_files(raw_dir: Path) -> list[tuple[int, int, Path]]:
    """Return (year, floor, path) for every CU-BEMS CSV in the folder, sorted."""
    found = []
    for path in sorted(raw_dir.glob("*.csv")):
        m = FILE_PATTERN.search(path.stem)
        if m:
            found.append((int(m.group(1)), int(m.group(2)), path))
    if not found:
        raise FileNotFoundError(f"No files like '2019Floor1.csv' found in {raw_dir.resolve()}")
    return found


def tidy_name(col: str) -> str:
    """'z1_AC1(kW)' -> 'z1_ac1_kw', 'z2_S1(degC)' -> 'z2_temp_c', 'z2_S1(RH%)' -> 'z2_rh_pct'."""
    m = COLUMN_PATTERN.match(col.strip())
    if not m:
        return col.strip().lower()
    zone, device, unit = m.groups()
    if unit == "kW":
        return f"{zone}_{device.lower()}_kw"
    # Sensor columns: drop the 'S1' when it is the zone's only sensor (true for all CU-BEMS files).
    prefix = zone if device.upper() == "S1" else f"{zone}_{device.lower()}"
    return {"degC": f"{prefix}_temp_c", "RH%": f"{prefix}_rh_pct", "lux": f"{prefix}_lux"}[unit]


def load_raw(path: Path) -> tuple[pd.DataFrame, pd.Timedelta, dict]:
    """Read one raw file onto a complete, regular time grid. Returns data, step and stats."""
    df = pd.read_csv(path)
    date_col = next((c for c in df.columns if c.strip().lower() in ("date", "datetime", "timestamp")), df.columns[0])
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    stats = {"rows_in_file": len(df), "bad_timestamps": int(df[date_col].isna().sum())}
    df = df.dropna(subset=[date_col])
    stats["duplicate_timestamps"] = int(df[date_col].duplicated().sum())
    df = df.drop_duplicates(subset=date_col, keep="first").set_index(date_col).sort_index()

    # Numbers only, stored as float32 to halve memory (2019 files have 525,600 rows).
    df = df.apply(pd.to_numeric, errors="coerce").astype("float32")

    # Infer the sampling step (1 minute for the original files) and fill in missing timestamps.
    step = df.index.to_series().diff().median()
    full_index = pd.date_range(df.index.min(), df.index.max(), freq=step)
    stats["missing_timestamps"] = int(len(full_index.difference(df.index)))
    df = df.reindex(full_index)
    df.index.name = "timestamp"

    df.columns = [tidy_name(c) for c in df.columns]
    if df.columns.duplicated().any():
        raise ValueError(f"Duplicate column names after renaming in {path.name}")
    return df, step, stats


def column_kind(col: str) -> str:
    if col.endswith("_kw"):
        return "power"
    if col.endswith("_temp_c"):
        return "temp"
    if col.endswith("_rh_pct"):
        return "rh"
    if col.endswith("_lux"):
        return "lux"
    return "other"

## 3. Helper functions: cleaning
Every rule writes a line to the cleaning log, so the methods slide can say exactly
what was changed and how much.

In [6]:
def run_lengths(mask: pd.Series) -> pd.Series:
    """For each position, the length of the run of equal mask values it belongs to."""
    groups = (mask != mask.shift()).cumsum()
    return mask.groupby(groups).transform("size")


def flatline_mask(s: pd.Series, min_steps: int) -> pd.Series:
    """True where a sensor repeats exactly the same value for at least `min_steps` readings."""
    groups = (s != s.shift()).cumsum()      # NaN != NaN, so gaps never count as flatlines
    sizes = s.groupby(groups).transform("size")
    return s.notna() & (sizes >= min_steps)


def fill_short_gaps(s: pd.Series, max_steps: int) -> tuple[pd.Series, int]:
    """Linearly interpolate gaps of at most `max_steps` readings; leave longer gaps empty."""
    missing = s.isna()
    if not missing.any():
        return s, 0
    fillable = missing & (run_lengths(missing) <= max_steps)
    if not fillable.any():
        return s, 0
    interpolated = s.interpolate(method="linear", limit_area="inside")
    filled = s.where(~fillable, interpolated)
    return filled, int((fillable & filled.notna()).sum())


def clean_frame(df: pd.DataFrame, step: pd.Timedelta, source: str, log: list) -> pd.DataFrame:
    """Apply every cleaning rule to one raw file. Mutates and returns df."""
    flat_steps = max(int(pd.Timedelta(hours=FLATLINE_HOURS) / step), 2)
    gap_steps = max(int(pd.Timedelta(minutes=SHORT_GAP_MINUTES) / step), 0)

    def note(column, rule, n, detail=""):
        if n:
            log.append({"source": source, "column": column, "rule": rule,
                        "readings_changed": int(n), "detail": detail})

    for col in df.columns:
        kind = column_kind(col)
        s = df[col]

        if kind == "power":
            negative = s < 0
            note(col, "negative power removed", negative.sum())
            s = s.mask(negative)
            cap = max(float(s.quantile(POWER_CAP_QUANTILE)) * POWER_CAP_MULTIPLIER, POWER_CAP_FLOOR_KW)
            spikes = s > cap
            note(col, "power spike removed", spikes.sum(), f"cap = {cap:.2f} kW")
            s = s.mask(spikes)

        elif kind in ("temp", "rh", "lux"):
            lo, hi = {"temp": TEMP_RANGE_C, "rh": RH_RANGE_PCT, "lux": LUX_RANGE}[kind]
            out_of_range = (s < lo) | (s > hi)
            note(col, "out-of-range reading removed", out_of_range.sum(), f"valid range {lo}-{hi}")
            s = s.mask(out_of_range)
            if kind in ("temp", "rh"):
                # Lux is legitimately constant (0) all night, so only T and RH are checked for flatlines.
                stuck = flatline_mask(s, flat_steps)
                note(col, "stuck sensor (flatline) removed", stuck.sum(), f">= {FLATLINE_HOURS} h unchanged")
                s = s.mask(stuck)

        if kind != "other" and gap_steps > 0:
            s, n_filled = fill_short_gaps(s, gap_steps)
            note(col, "short gap interpolated", n_filled, f"gaps <= {SHORT_GAP_MINUTES} min")

        df[col] = s
    return df

## 4. Helper functions: aggregation and derived columns

In [7]:
def aggregate(df: pd.DataFrame, rule: str, step: pd.Timedelta) -> pd.DataFrame:
    """Average to `rule` (e.g. '15min', 'h'), keeping a window only if enough readings exist.

    Each value is labelled with the START of its window (10:00 = 10:00-10:59 for hourly).
    For power columns, the hourly mean kW equals the kWh used in that hour.
    """
    expected = pd.Timedelta(rule if rule[0].isdigit() else "1" + rule) / step
    needed = max(int(np.ceil(expected * MIN_COVERAGE)), 1)
    means = df.resample(rule).mean()
    counts = df.resample(rule).count()
    return means.where(counts >= needed)


def humidity_ratio(temp_c: pd.Series, rh_pct: pd.Series, pressure_kpa: float = 101.325) -> pd.Series:
    """Humidity ratio (kg water / kg dry air), for the ASHRAE 55 limit of 0.012."""
    p_ws = 0.61094 * np.exp(17.625 * temp_c / (temp_c + 243.04))   # saturation pressure, kPa
    p_w = rh_pct / 100.0 * p_ws
    return 0.622 * p_w / (pressure_kpa - p_w)


def idle_ac_columns(df: pd.DataFrame) -> list[str]:
    """AC units that average below IDLE_AC_MEAN_KW: effectively never on (likely decommissioned)."""
    ac = [c for c in df.columns if re.match(r"^z\d+_ac\d+_kw$", c)]
    return [c for c in ac if not (df[c].mean() >= IDLE_AC_MEAN_KW)]


def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Zone AC totals, floor totals by end use, and indoor humidity ratio.

    Idle AC units are left out of the totals: they contribute ~0 kW, but their data gaps
    would otherwise blank out the total for the whole zone or floor.
    """
    new = {}
    idle = set(idle_ac_columns(df))
    zones = sorted({c.split("_")[0] for c in df.columns if re.match(r"^z\d+_", c)}, key=lambda z: int(z[1:]))
    for z in zones:
        ac_cols = [c for c in df.columns if re.match(rf"^{z}_ac\d+_kw$", c) and c not in idle]
        if ac_cols:
            # Target for the Question B baseline. NaN if any unit is missing, so totals are never understated.
            new[f"{z}_ac_total_kw"] = df[ac_cols].sum(axis=1, min_count=len(ac_cols))
        if f"{z}_temp_c" in df and f"{z}_rh_pct" in df:
            new[f"{z}_humidity_ratio"] = humidity_ratio(df[f"{z}_temp_c"], df[f"{z}_rh_pct"])

    end_uses = {
        "floor_ac_kw": [c for c in df.columns if re.match(r"^z\d+_ac\d+_kw$", c) and c not in idle],
        "floor_light_kw": [c for c in df.columns if re.match(r"^z\d+_light_kw$", c)],
        "floor_plug_kw": [c for c in df.columns if re.match(r"^z\d+_plug_kw$", c)],
    }
    for name, cols in end_uses.items():
        if cols:
            new[name] = df[cols].sum(axis=1, min_count=len(cols))
    parts = [new[k] for k in end_uses if k in new]
    if parts:
        new["floor_total_kw"] = pd.concat(parts, axis=1).sum(axis=1, min_count=len(parts))
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

## 5. Weather from Open-Meteo
One request covers the whole period. The result is saved to `weather/openmeteo_bangkok_hourly.csv`,
so later runs read the file instead of calling the API again. Delete that file to re-download.

In [8]:
def fetch_weather(cache_path: Path) -> pd.DataFrame:
    """Hourly Bangkok weather for the study period, labelled in local time."""
    if cache_path.exists():
        print(f"Weather: using cached file {cache_path}")
        return pd.read_csv(cache_path, parse_dates=["timestamp"], index_col="timestamp")

    # Ask for one extra day so the preceding-hour variables can be re-labelled at the end.
    end_plus_one = (pd.Timestamp(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": START_DATE,
        "end_date": end_plus_one,
        "hourly": ",".join(WEATHER_VARS),
        "timezone": TIMEZONE,
        "wind_speed_unit": "kmh",
    }
    print("Weather: downloading from Open-Meteo ...")
    for attempt in range(1, 4):
        try:
            response = requests.get(OPEN_METEO_URL, params=params, timeout=120)
            response.raise_for_status()
            payload = response.json()
            break
        except (requests.RequestException, ValueError) as err:
            if attempt == 3:
                raise RuntimeError(f"Open-Meteo request failed after 3 attempts: {err}") from err
            print(f"  attempt {attempt} failed ({err}); retrying ...")
            time.sleep(5 * attempt)

    if "hourly" not in payload:
        raise RuntimeError(f"Unexpected Open-Meteo response: {str(payload)[:300]}")
    if payload.get("utc_offset_seconds") != 7 * 3600:
        print(f"  WARNING: UTC offset is {payload.get('utc_offset_seconds')} s, expected 25200 s (UTC+7).")

    w = pd.DataFrame(payload["hourly"])
    w["time"] = pd.to_datetime(w["time"])
    w = w.set_index("time").rename(columns=WEATHER_VARS)
    w.index.name = "timestamp"
    for col in PRECEDING_HOUR_VARS:
        w[col] = w[col].shift(-1)       # value stamped 11:00 describes 10:00-11:00
    w = w.loc[START_DATE:END_DATE]
    print("  units:", payload.get("hourly_units"))

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    w.to_csv(cache_path)
    print(f"  saved {len(w):,} hourly rows to {cache_path}")
    return w


def weather_to_15min(w: pd.DataFrame) -> pd.DataFrame:
    """Hourly weather -> 15-min: interpolate instantaneous variables, hold hourly totals constant."""
    idx = pd.date_range(w.index.min(), w.index.max() + pd.Timedelta(minutes=45), freq="15min")
    instant = [c for c in w.columns if c not in PRECEDING_HOUR_VARS]
    out = w[instant].reindex(idx).interpolate(method="time", limit_area="inside").ffill(limit=3)
    block = w[[c for c in PRECEDING_HOUR_VARS if c in w]].reindex(idx).ffill(limit=3)
    out = pd.concat([out, block], axis=1)
    out.index.name = "timestamp"
    return out


def weather_hourly_means(w15: pd.DataFrame) -> pd.DataFrame:
    """Hour-average weather (mean of the four 15-min values) to match hourly building means."""
    out = w15.resample("h").mean()
    out["out_cdh"] = (out["out_temp_c"] - CDD_BASE_C).clip(lower=0)   # cooling degree hours
    return out


def daily_cdd(w: pd.DataFrame) -> pd.DataFrame:
    daily = w["out_temp_c"].resample("D").mean().to_frame("out_temp_daily_mean_c")
    daily["cdd"] = (daily["out_temp_daily_mean_c"] - CDD_BASE_C).clip(lower=0)
    return daily

## 6. Calendar: Thai public holidays and occupied hours

In [9]:
def build_holidays() -> pd.DataFrame:
    years = list(range(pd.Timestamp(START_DATE).year, pd.Timestamp(END_DATE).year + 1))
    try:
        th = holidays.country_holidays("TH", years=years, language="en_US")
    except Exception:  # older versions of the package without English names
        th = holidays.country_holidays("TH", years=years)
    hol = pd.DataFrame(sorted(th.items()), columns=["date", "holiday_name"])
    if EXTRA_HOLIDAYS:
        extra = pd.DataFrame(list(EXTRA_HOLIDAYS.items()), columns=["date", "holiday_name"])
        hol = pd.concat([hol, extra], ignore_index=True)
    hol["date"] = pd.to_datetime(hol["date"])
    hol = hol.groupby("date", as_index=False)["holiday_name"].agg("; ".join)
    return hol[(hol["date"] >= START_DATE) & (hol["date"] <= END_DATE)].reset_index(drop=True)


def calendar_columns(index: pd.DatetimeIndex, hol: pd.DataFrame) -> pd.DataFrame:
    day = index.normalize()
    names = hol.set_index("date")["holiday_name"]
    is_holiday = day.isin(names.index)
    is_weekend = index.dayofweek >= 5
    is_workday = ~is_weekend & ~is_holiday
    prev_day_holiday = (day - pd.Timedelta(days=1)).isin(names.index)
    cal = pd.DataFrame({
        "date": day,
        "year": index.year,
        "month": index.month,
        "hour": index.hour,
        "day_of_week": index.dayofweek,            # 0 = Monday
        "day_name": index.day_name(),
        "is_weekend": is_weekend,
        "is_holiday": is_holiday,
        "holiday_name": pd.Series(day, index=index).map(names).fillna("").values,
        "day_type": np.select([is_holiday, is_weekend], ["holiday", "weekend"], "workday"),
        "is_workday": is_workday,
        "day_after_holiday": is_workday & prev_day_holiday,
        "is_occupied": is_workday & (index.hour >= OCC_START_HOUR) & (index.hour < OCC_END_HOUR),
    }, index=index)
    return cal

## 7. Reports

In [10]:
def quality_rows(source: str, raw: pd.DataFrame, clean: pd.DataFrame, hourly: pd.DataFrame) -> list[dict]:
    rows = []
    for col in raw.columns:
        rows.append({
            "source": source,
            "column": col,
            "kind": column_kind(col),
            "pct_missing_raw": round(raw[col].isna().mean() * 100, 2),
            "pct_missing_after_cleaning": round(clean[col].isna().mean() * 100, 2),
            "pct_missing_hourly": round(hourly[col].isna().mean() * 100, 2),
            "mean_after_cleaning": round(float(clean[col].mean()), 3) if clean[col].notna().any() else np.nan,
            "max_raw": round(float(raw[col].max()), 3) if raw[col].notna().any() else np.nan,
            "max_after_cleaning": round(float(clean[col].max()), 3) if clean[col].notna().any() else np.nan,
        })
    return rows


def save_table(df: pd.DataFrame, path: Path, index: bool = True) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index, float_format="%.4f")
    try:  # Parquet is much faster to reload; skipped quietly if pyarrow isn't installed
        df.to_parquet(path.with_suffix(".parquet"), index=index)
    except ImportError:
        pass


def timestamp_check(hourly: pd.DataFrame, label: str) -> None:
    """Daylight check: indoor lux should peak around midday if timestamps are correct."""
    lux_cols = [c for c in hourly.columns if c.endswith("_lux")]
    if not lux_cols:
        return
    by_hour = hourly[lux_cols].mean(axis=1).groupby(hourly.index.hour).mean()
    if by_hour.notna().any():
        peak = int(by_hour.idxmax())
        status = "OK" if 8 <= peak <= 16 else "CHECK TIMESTAMPS"
        print(f"  {label}: lux peaks at {peak:02d}:00 -> {status}")

## 8. Run the pipeline

In [11]:
def main() -> dict:
    for sub in ["clean_15min", "clean_hourly", "weather", "calendar", "reports"]:
        (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

    # Shared inputs: weather and holidays
    weather = fetch_weather(OUT_DIR / "weather" / "openmeteo_bangkok_hourly.csv")
    w15 = weather_to_15min(weather)
    w_hourly = weather_hourly_means(w15)
    save_table(daily_cdd(weather), OUT_DIR / "weather" / "daily_weather_cdd.csv")
    hol = build_holidays()
    save_table(hol, OUT_DIR / "calendar" / "thai_holidays.csv", index=False)
    print(f"Holidays: {len(hol)} Thai public holidays between {START_DATE} and {END_DATE}")

    files = find_raw_files(RAW_DIR)
    print(f"Found {len(files)} raw files in {RAW_DIR}")

    cleaning_log, quality, file_stats = [], [], []
    by_floor: dict[int, dict[str, list[pd.DataFrame]]] = {}

    for year, floor, path in files:
        source = path.name
        t0 = time.time()
        raw, step, stats = load_raw(path)
        clean = clean_frame(raw.copy(), step, source, cleaning_log)
        q15 = aggregate(clean, "15min", step)
        q60 = aggregate(clean, "h", step)
        quality += quality_rows(source, raw, clean, q60)
        file_stats.append({"source": source, "year": year, "floor": floor, "step": str(step),
                           "start": raw.index.min(), "end": raw.index.max(), **stats})
        by_floor.setdefault(floor, {"15min": [], "hourly": []})
        by_floor[floor]["15min"].append(q15)
        by_floor[floor]["hourly"].append(q60)
        print(f"  {source}: {len(raw):,} rows at {step}, cleaned in {time.time() - t0:.0f} s")
        del raw, clean

    # Join years per floor, add derived columns, weather and calendar, then save.
    hourly_by_floor = {}
    for floor in sorted(by_floor):
        for res, wx in (("15min", w15), ("hourly", w_hourly)):
            data = pd.concat(by_floor[floor][res]).sort_index()
            data = data[~data.index.duplicated(keep="first")]
            data = add_derived(data)
            out = pd.concat([calendar_columns(data.index, hol), data, wx.reindex(data.index)], axis=1)
            out.index.name = "timestamp"
            save_table(out, OUT_DIR / f"clean_{res}" / f"floor{floor}_{res}.csv")
            if res == "hourly":
                hourly_by_floor[floor] = out
                timestamp_check(out, f"Floor {floor}")
        print(f"Floor {floor}: saved 15-min and hourly tables "
              f"({hourly_by_floor[floor].index.min():%Y-%m-%d} to {hourly_by_floor[floor].index.max():%Y-%m-%d})")

    # Building-wide hourly table: building columns prefixed by floor, calendar and weather once.
    shared = [c for c in next(iter(hourly_by_floor.values())).columns
              if c in w_hourly.columns or c in calendar_columns(pd.DatetimeIndex([]), hol).columns]
    full_index = pd.DatetimeIndex(sorted(set().union(*[df.index for df in hourly_by_floor.values()])))
    blocks = [calendar_columns(full_index, hol), w_hourly.reindex(full_index)]
    for floor, df in hourly_by_floor.items():
        blocks.append(df.drop(columns=shared).add_prefix(f"f{floor}_").reindex(full_index))
    all_floors = pd.concat(blocks, axis=1)
    all_floors.index.name = "timestamp"
    save_table(all_floors, OUT_DIR / "all_floors_hourly.csv")

    # Reports
    log_df = pd.DataFrame(cleaning_log, columns=["source", "column", "rule", "readings_changed", "detail"])
    quality_df = pd.DataFrame(quality)
    idle = quality_df[(quality_df["column"].str.match(r"^z\d+_ac\d+_kw$"))
                      & (quality_df["mean_after_cleaning"] < IDLE_AC_MEAN_KW)]
    save_table(log_df, OUT_DIR / "reports" / "cleaning_log.csv", index=False)
    save_table(quality_df, OUT_DIR / "reports" / "data_quality.csv", index=False)
    save_table(pd.DataFrame(file_stats), OUT_DIR / "reports" / "file_summary.csv", index=False)
    save_table(idle[["source", "column", "mean_after_cleaning"]], OUT_DIR / "reports" / "idle_units.csv", index=False)

    print("\nCleaning summary (readings changed, all files):")
    if len(log_df):
        print(log_df.groupby("rule")["readings_changed"].sum().sort_values(ascending=False).to_string())
    print(f"\nIdle AC units flagged: {len(idle)} (kept in the data, listed in reports/idle_units.csv)")
    wx_cov = all_floors["out_temp_c"].notna().mean() * 100
    print(f"Weather coverage in hourly table: {wx_cov:.1f}%")
    print(f"\nDone. Outputs in {OUT_DIR.resolve()}")
    return {"hourly": hourly_by_floor, "all_floors": all_floors, "log": log_df,
            "quality": quality_df, "holidays": hol, "weather": weather}


results = main()

Weather: downloading from Open-Meteo ...
  units: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'dew_point_2m': '°C', 'apparent_temperature': '°C', 'precipitation': 'mm', 'cloud_cover': '%', 'shortwave_radiation': 'W/m²', 'wind_speed_10m': 'km/h'}
  saved 13,176 hourly rows to ..\..\data\qns2\processed\weather\openmeteo_bangkok_hourly.csv
Holidays: 36 Thai public holidays between 2018-07-01 and 2019-12-31
Found 14 raw files in ..\..\data\qns2
  2018Floor1.csv: 264,960 rows at 0 days 00:01:00, cleaned in 3 s
  2018Floor2.csv: 264,960 rows at 0 days 00:01:00, cleaned in 6 s
  2018Floor3.csv: 264,960 rows at 0 days 00:01:00, cleaned in 5 s
  2018Floor4.csv: 264,960 rows at 0 days 00:01:00, cleaned in 3 s
  2018Floor5.csv: 264,960 rows at 0 days 00:01:00, cleaned in 5 s
  2018Floor6.csv: 264,960 rows at 0 days 00:01:00, cleaned in 5 s
  2018Floor7.csv: 264,960 rows at 0 days 00:01:00, cleaned in 5 s
  2019Floor1.csv: 525,600 rows at 0 days 00:01:00, cleaned in 4 

## 9. Quick look (optional)
After the run, these lines give a first view of the Question B data for Floor 4.

```python
f4 = results["hourly"][4]
occ = f4[f4["is_occupied"]]
print(occ[["z1_ac_total_kw", "z2_ac_total_kw", "z4_ac_total_kw", "z5_ac_total_kw", "out_temp_c"]].describe())
print(occ.groupby("month")[["z4_temp_c", "out_temp_c"]].mean())
```